In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Lipinski, Descriptors

if not os.path.exists('sascorer.py'):
    !wget https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py
if not os.path.exists('fpscores.pkl.gz'):
    !wget https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz

import sascorer

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
ADMET_PATH  = Path("../data/link_invent_outputs/admet_test_protac_optimized_full.csv")
SCORED_PATH = Path("../data/link_invent_outputs/scored_sampling_test_protac_optimized_full.csv")
OUTPUT_ALL  = Path("../data/link_invent_outputs/protac_ranked.csv")
OUTPUT_TOP = Path("../data/link_invent_outputs/protac_top_docking.csv")

In [3]:
# ── 1. Load & Merge ─────────────────────────────────────────────────────────
admet  = pd.read_csv(ADMET_PATH)
scored = pd.read_csv(SCORED_PATH)

df = pd.merge(admet, scored[["SMILES", "Active_Class", "Active_Probability",
                              "Epistemic_Uncertainty"]],
              on="SMILES", how="inner")
print(f"Merged dataset: {len(df)} compounds")

Merged dataset: 703 compounds


In [4]:
# ── 2. Нормалізація alert-колонок (float → int) ─────────────────────────────
for col in ["PAINS_alert", "BRENK_alert", "AMES", "hERG"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

In [5]:
# ── 3. Лінкерні дескриптори (RDKit) ─────────────────────────────────────────
def analyze_linker(smi):
    if pd.isna(smi):
        return None, None, None
    mol = Chem.MolFromSmiles(smi)
    if not mol:
        return None, None, None
    return (mol.GetNumHeavyAtoms(),
            Lipinski.FractionCSP3(mol),
            Descriptors.NumRotatableBonds(mol))

df[["Linker_Length", "Linker_fCsp3", "Linker_RotBonds"]] = df["Linker"].apply(
    lambda x: pd.Series(analyze_linker(x))
)
print(f"Linker descriptors computed. NaN rows: {df['Linker_Length'].isna().sum()}")

Linker descriptors computed. NaN rows: 0


In [6]:
# ── 4. SA Score (RDKit sascorer) ─────────────────────────────────────────────
def calc_sa(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(smi)
    return sascorer.calculateScore(mol) if mol else None

df["SA_Score"] = df["SMILES"].apply(calc_sa)
print(f"SA Score computed. NaN rows: {df['SA_Score'].isna().sum()}")

SA Score computed. NaN rows: 0


In [7]:
# ── 5. Hard Filters ──────────────────────────────────────────────────────────
filters = {
    "hERG"        : df["hERG"]         == 0,
    "AMES"        : df["AMES"]         == 0,
    "PAINS"       : df["PAINS_alert"]  == 0,
    "LogP"        : df["logP"]         <= 8.0,
    "Linker_Len"  : df["Linker_Length"].between(6, 18),
    "Linker_fCsp3": df["Linker_fCsp3"] >= 0.4,
    "Linker_Rot"  : df["Linker_RotBonds"] <= 10,
    "Active"      : df["Active_Probability"] >= 0.70,
    "Epistemic"   : df["Epistemic_Uncertainty"] < 0.35,
}

# Діагностика
print(f"\n── Filter diagnostics (n={len(df)}) ──")
cumulative = pd.Series(True, index=df.index)
for name, cond in filters.items():
    cumulative &= cond
    print(f"  {name:20s}  pass={cond.sum():4d}  cumulative={cumulative.sum():4d}")

df_f = df[cumulative].copy()
print(f"\nAfter hard filters: {len(df_f)} compounds")


── Filter diagnostics (n=703) ──
  hERG                  pass= 703  cumulative= 703
  AMES                  pass= 703  cumulative= 703
  PAINS                 pass= 689  cumulative= 689
  LogP                  pass= 695  cumulative= 681
  Linker_Len            pass= 695  cumulative= 675
  Linker_fCsp3          pass= 697  cumulative= 670
  Linker_Rot            pass= 703  cumulative= 670
  Active                pass= 538  cumulative= 518
  Epistemic             pass= 703  cumulative= 518

After hard filters: 518 compounds


In [8]:
# ── 6. Нормалізація ──────────────────────────────────────────────────────────
def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else pd.Series(0.5, index=s.index)

def invert(s):
    return 1 - minmax(s)

df_f["norm_active_prob"] = minmax(df_f["Active_Probability"])
df_f["norm_caco2"]       = minmax(df_f["Caco2_Wang"])
df_f["norm_sa"]          = invert(df_f["SA_Score"])        # менше = синтетично доступніше
df_f["norm_logp"]        = np.exp(-0.5 * ((df_f["logP"] - 5.5) / 1.5) ** 2)  # Gaussian, оптимум 5.5
df_f["norm_frcsp3"]      = minmax(df_f["Linker_fCsp3"])
df_f["norm_uncertainty"] = invert(df_f["Epistemic_Uncertainty"])
df_f["norm_brenk"]       = (df_f["BRENK_alert"] == 0).astype(float)  # soft penalty

In [ ]:
# ── 7. Composite Score ───────────────────────────────────────────────────────
weights = {
    "norm_active_prob" : 0.30,
    "norm_caco2"       : 0.22,
    "norm_sa"          : 0.18, 
    "norm_logp"        : 0.10,
    "norm_frcsp3"      : 0.10,
    "norm_uncertainty" : 0.07,
    "norm_brenk"       : 0.03,
}
# Сума ваг = 1.0 ✓

df_f["composite_score"] = sum(w * df_f[col] for col, w in weights.items())

# Score breakdown (для інтерпретації)
for col, w in weights.items():
    df_f[f"contrib_{col}"] = w * df_f[col]

In [10]:
# ── 8. Ранжування ────────────────────────────────────────────────────────────
df_ranked = df_f.sort_values("composite_score", ascending=False).reset_index(drop=True)
df_ranked["rank"] = df_ranked.index + 1

# Вихідні колонки (оригінальні значення + scoring)
out_cols = [
    "rank", "SMILES", "Warheads", "Linker",
    "composite_score",
    # Scoring компоненти (оригінальні значення)
    "Active_Probability", "Epistemic_Uncertainty",
    "Caco2_Wang", "SA_Score", "logP", "Linker_fCsp3",
    "BRENK_alert",
    # Лінкер
    "Linker_Length", "Linker_RotBonds",
    # Фіз-хім
    "molecular_weight", "tpsa", "hydrogen_bond_donors", "hydrogen_bond_acceptors",
    # Safety (hard filters)
    "hERG", "AMES", "PAINS_alert",
    # Link-INVENT
    "NLL",
    # Score contributions
    *[f"contrib_{c}" for c in weights],
]
out_cols = [c for c in out_cols if c in df_ranked.columns]
df_ranked[out_cols].to_csv(OUTPUT_ALL, index=False)
print(f"Saved {len(df_ranked)} ranked compounds → {OUTPUT_ALL}")

Saved 518 ranked compounds → ../data/link_invent_outputs/protac_ranked.csv


In [12]:
# ── 9. Top-5 Diverse (MaxMin Tanimoto) ──────────────────────────────────────
from rdkit.Chem import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

def diverse_top_n(df, smiles_col="SMILES", top_n=5, pool_size=150):
    pool = df.head(pool_size).copy()
    mols = [(i, Chem.MolFromSmiles(s)) for i, s in zip(pool.index, pool[smiles_col])]
    mols = [(i, m) for i, m in mols if m]
    gen  = GetMorganGenerator(radius=2, fpSize=2048)
    fps  = {i: gen.GetFingerprint(m) for i, m in mols}

    selected = [mols[0][0]]
    while len(selected) < top_n:
        remaining = [i for i, _ in mols if i not in selected]
        if not remaining:
            break
        most_diverse = max(
            remaining,
            key=lambda i: min(DataStructs.TanimotoSimilarity(fps[i], fps[s])
                              for s in selected)
        )
        sim = min(DataStructs.TanimotoSimilarity(fps[most_diverse], fps[s]) for s in selected)
        print(f"  Selected rank={pool.loc[most_diverse,'rank']}  "
              f"min_tanimoto={sim:.3f}  score={pool.loc[most_diverse,'composite_score']:.4f}")
        selected.append(most_diverse)

    result = pool.loc[selected, out_cols].copy()
    result["docking_priority"] = range(1, len(result) + 1)
    result.to_csv(OUTPUT_TOP, index=False)
    print(f"Saved top-{top_n} diverse candidates → {OUTPUT_TOP}")
    return result

print(f"\n── Selecting Top-5 Diverse Candidates (pool=top-150) ──")
df_top5 = diverse_top_n(df_ranked, top_n=5, pool_size=150)

print(f"\n{'='*65}")
print("TOP-5 DIVERSE CANDIDATES FOR PRIMARY DOCKING")
print(f"{'='*65}")
print(df_top5[["docking_priority", "composite_score",
               "Active_Probability", "Caco2_Wang",
               "SA_Score", "logP", "Linker_fCsp3"]].to_string(index=False))


── Selecting Top-5 Diverse Candidates (pool=top-150) ──
  Selected rank=27  min_tanimoto=0.912  score=0.6843
  Selected rank=19  min_tanimoto=0.784  score=0.6954
  Selected rank=2  min_tanimoto=0.781  score=0.7184
  Selected rank=60  min_tanimoto=0.753  score=0.6565
Saved top-5 diverse candidates → ../data/link_invent_outputs/protac_top_docking.csv

TOP-5 DIVERSE CANDIDATES FOR PRIMARY DOCKING
 docking_priority  composite_score  Active_Probability  Caco2_Wang  SA_Score    logP  Linker_fCsp3
                1         0.719623            0.761459   -5.268083  4.059620 5.51152      0.666667
                2         0.684302            0.743593   -5.220900  4.028466 5.31712      0.636364
                3         0.695386            0.747893   -5.215442  4.029066 5.31712      0.636364
                4         0.718391            0.763663   -5.296331  3.970738 5.20310      0.636364
                5         0.656549            0.766387   -5.438529  4.108678 6.45696      0.647059


In [14]:
# ── 9. Top Diverse via Linker Murcko Scaffold Clustering ──────────────────
from rdkit.Chem import DataStructs, Scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

def get_linker_scaffold(smi):
    """Murcko scaffold лінкера (не повного PROTACа)."""
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(smi)
    if not mol:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold) if scaffold else Chem.MolToSmiles(mol)

print(f"\n── Best candidate per Murcko scaffold ──")

pool = df_ranked.head(200).copy()
pool["linker_scaffold"] = pool["Linker"].apply(get_linker_scaffold)

# Найкращий за composite_score з кожного scaffold
best_per_scaffold = (pool
                     .groupby("linker_scaffold", sort=False)
                     .first()
                     .reset_index()
                     .sort_values("composite_score", ascending=False)
                     .reset_index(drop=True))

best_per_scaffold["docking_priority"] = best_per_scaffold.index + 1
print(f"  Unique scaffolds: {len(best_per_scaffold)}")

out = [c for c in out_cols + ["linker_scaffold"] if c in best_per_scaffold.columns]
best_per_scaffold[out].to_csv(OUTPUT_TOP, index=False)
print(f"Saved {len(best_per_scaffold)} candidates → {OUTPUT_TOP}")

display_cols = ["docking_priority", "linker_scaffold", "composite_score",
                "Active_Probability", "Caco2_Wang", "SA_Score", "logP", "Linker_fCsp3"]
display_cols = [c for c in display_cols if c in best_per_scaffold.columns]
print(f"\n{'='*75}")
print("BEST CANDIDATE PER SCAFFOLD (sorted by composite score)")
print(f"{'='*75}")
print(best_per_scaffold[display_cols].to_string(index=False))


── Best candidate per Murcko scaffold ──
  Unique scaffolds: 27
Saved 27 candidates → ../data/link_invent_outputs/protac_top_docking.csv

BEST CANDIDATE PER SCAFFOLD (sorted by composite score)
 docking_priority       linker_scaffold  composite_score  Active_Probability  Caco2_Wang  SA_Score    logP  Linker_fCsp3
                1     c1csc(C2CCCCC2)c1         0.719623            0.761459   -5.268083  4.059620 5.51152      0.666667
                2    c1ccc(C2CCCCC2)cc1         0.715781            0.784830   -5.320461  4.157505 6.51654      0.647059
                3  c1c[nH]c(C2CCCCC2)c1         0.701478            0.777970   -5.472687  4.144298 6.07292      0.733333
                4    c1ccn(CC2CCCCC2)c1         0.680591            0.769375   -5.508506  4.091869 4.89814      0.692308
                5  c1ccc(CN2CCCCCC2)cc1         0.670111            0.801849   -5.637017  4.122800 5.02540      0.571429
                6    c1ccc(N2CCCCC2)cc1         0.667976            0.785681   